# GRU Alarm Prediction — 5 Minutes Ahead
Binary time-series classification with precision-prioritized training, GroupKFold CV, and Optuna HPO.

## 0. Imports & Setup

In [1]:
import os, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_curve, auc, average_precision_score,
    ConfusionMatrixDisplay
)

import optuna
from optuna.samplers import TPESampler

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Reproducibility 
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print(f'Using device: {DEVICE}')

Using device: cpu


/Users/sanjana/Desktop/Training with selected features/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Data

In [ ]:
# ── !! EDIT THESE PATHS !! ─────────────────────────────────────────────────────
DATA_PATH     = '../Data/data.csv'
FEATURES_PATH = '../Data/selected_features.csv'
# ──────────────────────────────────────────────────────────────────────────────

# Load selected feature names
feat_df = pd.read_csv(FEATURES_PATH)
SELECTED_FEATURES = feat_df.iloc[:, 0].tolist()
print(f'Selected features to create: {len(SELECTED_FEATURES)}')

# Load main data
df = pd.read_csv(DATA_PATH)
print(f'Data shape (raw): {df.shape}')

# ── Create 'timestamp' column from 'TimeStamp' ──
df['timestamp'] = pd.to_datetime(df['TimeStamp']) 
df = df.drop(columns=['TimeStamp']).sort_values('timestamp').reset_index(drop=True)

# ── Identify base columns (raw sensor features) ──
BASE_COLS = [c for c in df.columns if c != 'timestamp']
print(f'Base sensor columns: {len(BASE_COLS)}')

# ── Feature Engineering Pipeline ──
def create_engineered_features(data, base_cols, selected_features=None):
    """Create only the engineered features required by selected_features."""
    df_eng = data.copy()
    
    if selected_features is None:
        selected_features = []
        
    # Pre-calculate row-wise features if needed
    if 'row_mean' in selected_features:
        df_eng['row_mean'] = data[base_cols].mean(axis=1)
    if 'row_abs_change' in selected_features:
        df_eng['row_abs_change'] = data[base_cols].diff(axis=1).abs().mean(axis=1)
        
    for feat in selected_features:
        if feat in df_eng.columns or feat == 'timestamp':
            continue
            
        if '_roll_min_' in feat:
            col, win = feat.split('_roll_min_')
            df_eng[feat] = data[col].rolling(window=int(win), min_periods=1).min()
        elif '_roll_max_' in feat:
            col, win = feat.split('_roll_max_')
            df_eng[feat] = data[col].rolling(window=int(win), min_periods=1).max()
        elif '_roll_range_' in feat:
            col, win = feat.split('_roll_range_')
            win = int(win)
            df_eng[feat] = (data[col].rolling(window=win, min_periods=1).max() - 
                            data[col].rolling(window=win, min_periods=1).min())
        elif '_roll_std_' in feat:
            col, win = feat.split('_roll_std_')
            df_eng[feat] = data[col].rolling(window=int(win), min_periods=1).std()
        elif '_roll_delta_' in feat:
            col, win = feat.split('_roll_delta_')
            win = int(win)
            df_eng[feat] = data[col] - data[col].shift(win - 1).bfill()
        elif '_ewm_std_' in feat:
            col, win = feat.split('_ewm_std_')
            df_eng[feat] = data[col].ewm(span=int(win), adjust=False).std()
        elif '_pct_change_' in feat:
            col, win = feat.split('_pct_change_')
            df_eng[feat] = data[col].pct_change(periods=int(win)).fillna(0)
        elif '_diff_' in feat:
            col, win = feat.split('_diff_')
            df_eng[feat] = data[col].diff(periods=int(win)).fillna(0)
            
    return df_eng

print('Creating engineered features (this may take a minute)...')
df = create_engineered_features(df, BASE_COLS, SELECTED_FEATURES)
print(f'Data shape (after engineering): {df.shape}')

# ── Keep only selected features + timestamp ──
cols_to_keep = ['timestamp'] + SELECTED_FEATURES
missing_cols = [c for c in cols_to_keep if c not in df.columns]
if missing_cols:
    print(f'Warning: Missing columns {missing_cols}')
df = df[[c for c in cols_to_keep if c in df.columns]]
print(f'Final feature set shape: {df.shape}')

# ── Create 'alarm' label based on statistical anomalies ──
from scipy import stats
X_features = df[SELECTED_FEATURES].values
z_scores = np.abs(stats.zscore(X_features, axis=0, nan_policy='omit'))
anomaly_flags = (z_scores > 2.5).sum(axis=1) > 5  # > 5 features with extreme z-score
df['alarm'] = anomaly_flags.astype(int)

# ── Create 'alarm_group' to group consecutive alarm events ──
alarm_transitions = df['alarm'].astype(int).diff().fillna(0).ne(0)
df['alarm_group'] = alarm_transitions.cumsum()

print(f'\n✓ Data shape (final): {df.shape}')
print(f'✓ Alarm distribution: {df["alarm"].value_counts().to_dict()}')
print(f'✓ Number of alarm groups: {df["alarm_group"].nunique()}')
print(f'✓ Column count: {len(df.columns)}')
df.head()

Selected features to create: 93
Data shape (raw): (1737585, 14)
Base sensor columns: 13
Creating engineered features (this may take a minute)...


KeyboardInterrupt: 

## 2. Configuration

In [ ]:
# ── !! EDIT COLUMN NAMES TO MATCH YOUR DATA !! ────────────────────────────────
TARGET_COL      = 'alarm'           # Binary label column (0/1) — created from anomaly detection
GROUP_COL       = 'alarm_group'     # Column identifying alarm group membership — from consecutive alarms
TIMESTAMP_COL   = 'timestamp'       # Timestamp column (used for ordering) — from 'TimeStamp'
# ──────────────────────────────────────────────────────────────────────────────

# Feature columns are now SELECTED_FEATURES (from the previous cell)
FEATURE_COLS = SELECTED_FEATURES

# Sequence config
LOOKBACK   = 20       # timesteps in each input window
STRIDE     = 1        # sliding window stride

# Training config
N_FOLDS    = 10
EPOCHS     = 100
PATIENCE   = 10       # early stopping
BATCH_SIZE = 32

# Precision-focus
POS_WEIGHT_DEFAULT = 0.7   # lower → model is more conservative about predicting alarm
THRESHOLD_DEFAULT  = 0.60  # classification threshold

# HPO
N_OPTUNA_TRIALS = 30

print(f'Using {len(FEATURE_COLS)} features')
print(f'LOOKBACK={LOOKBACK}, STRIDE={STRIDE}, N_FOLDS={N_FOLDS}')

Using 93 features
LOOKBACK=20, STRIDE=1, N_FOLDS=10


## 3. Split into Train / Validation / Test by Alarm Group

In [ ]:
# Sort by timestamp so sequences are chronological within each group
if TIMESTAMP_COL in df.columns:
    df = df.sort_values([GROUP_COL, TIMESTAMP_COL]).reset_index(drop=True)

groups = df[GROUP_COL].unique()
np.random.shuffle(groups)

n_total = len(groups)
n_train = int(n_total * 0.70)
n_val   = int(n_total * 0.15)

train_groups = groups[:n_train]
val_groups   = groups[n_train:n_train + n_val]
test_groups  = groups[n_train + n_val:]

df_train = df[df[GROUP_COL].isin(train_groups)].reset_index(drop=True)
df_val   = df[df[GROUP_COL].isin(val_groups)].reset_index(drop=True)
df_test  = df[df[GROUP_COL].isin(test_groups)].reset_index(drop=True)

print(f'Total groups : {n_total}')
print(f'Train groups : {len(train_groups)} | rows: {len(df_train)}')
print(f'Val groups   : {len(val_groups)}   | rows: {len(df_val)}')
print(f'Test groups  : {len(test_groups)}  | rows: {len(df_test)}')

Total groups : 68743
Train groups : 48120 | rows: 1228497
Val groups   : 10311   | rows: 262903
Test groups  : 10312  | rows: 246185


## 4. Sequence Dataset

In [ ]:
def build_sequences(data: pd.DataFrame, feature_cols, target_col, group_col,
                    lookback: int, stride: int):
    """
    Builds sliding-window sequences per alarm group so windows never
    cross group boundaries.
    Returns X (N, lookback, features), y (N,), groups (N,)
    """
    X_list, y_list, g_list = [], [], []
    for grp, grp_df in data.groupby(group_col):
        vals = grp_df[feature_cols].values.astype(np.float32)
        labels = grp_df[target_col].values.astype(np.float32)
        for i in range(0, len(vals) - lookback, stride):
            X_list.append(vals[i:i + lookback])
            y_list.append(labels[i + lookback - 1])  # label at end of window
            g_list.append(grp)
    return (np.array(X_list, dtype=np.float32),
            np.array(y_list, dtype=np.float32),
            np.array(g_list))


class AlarmDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# Build train sequences (val/test built separately after scaling)
X_train_raw, y_train, g_train = build_sequences(
    df_train, FEATURE_COLS, TARGET_COL, GROUP_COL, LOOKBACK, STRIDE)
X_val_raw,   y_val,   g_val   = build_sequences(
    df_val,   FEATURE_COLS, TARGET_COL, GROUP_COL, LOOKBACK, STRIDE)
X_test_raw,  y_test,  g_test  = build_sequences(
    df_test,  FEATURE_COLS, TARGET_COL, GROUP_COL, LOOKBACK, STRIDE)

print(f'Train sequences : {X_train_raw.shape}')
print(f'Val sequences   : {X_val_raw.shape}')
print(f'Test sequences  : {X_test_raw.shape}')
print(f'Class balance (train) — pos: {y_train.mean():.3f}')

Train sequences : (942616, 20, 93)
Val sequences   : (201029, 20, 93)
Test sequences  : (185845, 20, 93)
Class balance (train) — pos: 0.042


## 5. GRU Model Definition

In [ ]:
class GRUAlarmNet(nn.Module):
    def __init__(self, input_size, hidden_units, num_layers, dropout, rec_dropout):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_units,
            num_layers=num_layers,
            batch_first=True,
            dropout=rec_dropout if num_layers > 1 else 0.0,
        )
        # Gradually reduce hidden → 64 → 32 → 1
        mid = max(32, hidden_units // 2)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_units, mid),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(mid, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        _, h_n = self.gru(x)          # h_n: (num_layers, B, hidden)
        out = self.head(h_n[-1])      # last layer's hidden state
        return out.squeeze(-1)        # (B,) — raw logits


def make_model(trial_or_dict, input_size):
    """Construct model from Optuna trial or plain dict of params."""
    if isinstance(trial_or_dict, dict):
        p = trial_or_dict
    else:
        t = trial_or_dict
        p = {
            'hidden_units': t.suggest_categorical('hidden_units', [64, 128, 256]),
            'num_layers'  : t.suggest_int('num_layers', 1, 3),
            'dropout'     : t.suggest_categorical('dropout', [0.2, 0.3, 0.4, 0.5]),
            'rec_dropout' : 0.2,
        }
    return GRUAlarmNet(
        input_size=input_size,
        hidden_units=p['hidden_units'],
        num_layers=p['num_layers'],
        dropout=p['dropout'],
        rec_dropout=p.get('rec_dropout', 0.2),
    ).to(DEVICE)

## 6. Focal Loss

In [ ]:
class FocalLoss(nn.Module):
    """Binary Focal Loss with pos_weight for class imbalance."""
    def __init__(self, gamma=2.0, pos_weight=1.0):
        super().__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets,
            pos_weight=torch.tensor(self.pos_weight, device=logits.device),
            reduction='none'
        )
        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        focal_weight = (1 - pt) ** self.gamma
        return (focal_weight * bce).mean()

## 7. Train / Evaluate Helpers

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    all_probs, all_labels = [], []
    criterion = nn.BCEWithLogitsLoss()
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        total_loss += loss.item() * len(y_batch)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(y_batch.cpu().numpy())
    return (total_loss / len(loader.dataset),
            np.array(all_probs),
            np.array(all_labels))


def best_threshold_for_precision(probs, labels, min_recall=0.6):
    """Find threshold that maximises precision subject to recall >= min_recall."""
    precisions, recalls, thresholds = precision_recall_curve(labels, probs)
    # thresholds has len N-1; precisions/recalls have len N
    valid = [(p, r, t) for p, r, t in zip(precisions[:-1], recalls[:-1], thresholds)
             if r >= min_recall]
    if not valid:
        return 0.5  # fallback
    best = max(valid, key=lambda x: x[0])
    return best[2]


def pr_auc(probs, labels):
    return average_precision_score(labels, probs)

## 8. Best Hyperparameters

In [ ]:
# ── Data cleaning before scaling ─────────────────────────────────────────────

def clean_sequences(X):
    """Replace inf and very large values, then clip outliers."""
    X = X.copy()
    # Replace inf / -inf with nan
    X[~np.isfinite(X)] = np.nan
    # Fill nan with column median (across all timesteps and samples)
    N, T, F = X.shape
    X_flat = X.reshape(-1, F)
    col_medians = np.nanmedian(X_flat, axis=0)
    # Replace any remaining nan with column median
    for f in range(F):
        mask = ~np.isfinite(X_flat[:, f])
        if mask.any():
            X_flat[mask, f] = col_medians[f]
    X = X_flat.reshape(N, T, F)
    # Clip extreme outliers to 5th–95th percentile per feature
    X_flat = X.reshape(-1, F)
    lower = np.percentile(X_flat, 1, axis=0)
    upper = np.percentile(X_flat, 99, axis=0)
    X_flat = np.clip(X_flat, lower, upper)
    return X_flat.reshape(N, T, F).astype(np.float32)


print("Cleaning sequences...")
X_train_clean = clean_sequences(X_train_raw)
X_val_clean   = clean_sequences(X_val_raw)
X_test_clean  = clean_sequences(X_test_raw)

# Quick sanity check
assert np.all(np.isfinite(X_train_clean)), "Still has inf/nan after cleaning!"
print(f"  Train  — min: {X_train_clean.min():.4f}, max: {X_train_clean.max():.4f}")
print(f"  Val    — min: {X_val_clean.min():.4f},   max: {X_val_clean.max():.4f}")
print(f"  Test   — min: {X_test_clean.min():.4f},  max: {X_test_clean.max():.4f}")

# ── Scaling (fit on train only) ───────────────────────────────────────────────

INPUT_SIZE = X_train_clean.shape[2]  # = 94

scaler_global = StandardScaler()
N_tr, T, F = X_train_clean.shape
scaler_global.fit(X_train_clean.reshape(-1, F))

def scale_sequences(scaler, X):
    N, T, F = X.shape
    return scaler.transform(X.reshape(-1, F)).reshape(N, T, F).astype(np.float32)

X_train_sc = scale_sequences(scaler_global, X_train_clean)
X_val_sc   = scale_sequences(scaler_global, X_val_clean)
X_test_sc  = scale_sequences(scaler_global, X_test_clean)

# Final check after scaling
assert np.all(np.isfinite(X_train_sc)), "Scaling introduced inf/nan!"
print(f"\nScaled train — mean: {X_train_sc.mean():.4f}, std: {X_train_sc.std():.4f}")
print("Data ready for training ✓")

Cleaning sequences...
  Train  — min: -36.1420, max: 355.8733
  Val    — min: -36.0853,   max: 347.6487
  Test   — min: -36.1845,  max: 378.0753

Scaled train — mean: 0.0000, std: 1.0000
Data ready for training ✓


In [ ]:
# ── CHECKPOINT SAVE (run once after scaling succeeds) ─────────────────────────
import joblib, os

CKPT_DIR = '../checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

np.save(f'{CKPT_DIR}/X_train_sc.npy',  X_train_sc)
np.save(f'{CKPT_DIR}/X_val_sc.npy',    X_val_sc)
np.save(f'{CKPT_DIR}/X_test_sc.npy',   X_test_sc)
np.save(f'{CKPT_DIR}/y_train.npy',     y_train)
np.save(f'{CKPT_DIR}/y_val.npy',       y_val)
np.save(f'{CKPT_DIR}/y_test.npy',      y_test)
np.save(f'{CKPT_DIR}/g_train.npy',     g_train)
np.save(f'{CKPT_DIR}/g_val.npy',       g_val)
np.save(f'{CKPT_DIR}/g_test.npy',      g_test)

joblib.dump(scaler_global, f'{CKPT_DIR}/scaler_global.pkl')

# Save config so reload cell knows the shapes
import json
ckpt_meta = {
    'INPUT_SIZE'   : int(INPUT_SIZE),
    'LOOKBACK'     : LOOKBACK,
    'FEATURE_COLS' : FEATURE_COLS,
    'TARGET_COL'   : TARGET_COL,
    'GROUP_COL'    : GROUP_COL,
}
with open(f'{CKPT_DIR}/meta.json', 'w') as f:
    json.dump(ckpt_meta, f, indent=2)

print(f'Checkpoint saved to {CKPT_DIR}/')
print(f'  X_train_sc : {X_train_sc.shape}')
print(f'  X_val_sc   : {X_val_sc.shape}')
print(f'  X_test_sc  : {X_test_sc.shape}')

Checkpoint saved to ../checkpoints/
  X_train_sc : (942616, 20, 93)
  X_val_sc   : (201029, 20, 93)
  X_test_sc  : (185845, 20, 93)


In [ ]:
# ── Best Hyperparameters ─────────────────────────────────────
# We bypass hyperparameter tuning to avoid memory issues and long training times.
# Parameters are chosen to prevent overfitting on the large dataset (1M sequences),
# while prioritizing precision.

best_params = {
    'hidden_units': 64,
    'num_layers': 1,
    'dropout': 0.3,
    'rec_dropout': 0.2,
    'lr': 1e-3,
    'batch_size': 256,
    'pos_weight': 0.5  # Lower pos_weight favors precision over recall in Focal Loss
}

print(f'Using fixed hyperparameters: {best_params}')

# ── Save best_params so subsequent cells work ────────────────────────────
import json
import os

CKPT_DIR = '../checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

with open(f'{CKPT_DIR}/best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)
print(f'best_params saved to {CKPT_DIR}/best_params.json')


Python(12131) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Running Optuna (30 trials)...


  0%|          | 0/30 [00:00<?, ?it/s]

: 

## 9. 10-Fold GroupKFold Training with Best Hyperparameters

In [ ]:
gkf10 = GroupKFold(n_splits=N_FOLDS)

fold_results  = []
fold_histories = []  # loss curves per fold
best_models   = []   # saved state dicts

for fold_idx, (tr_idx, val_idx) in enumerate(
        gkf10.split(X_train_sc, y_train, g_train)):

    print(f'\n── Fold {fold_idx + 1}/{N_FOLDS} ──────────────────────')

    X_f_tr, y_f_tr   = X_train_sc[tr_idx], y_train[tr_idx]
    X_f_val, y_f_val = X_train_sc[val_idx], y_train[val_idx]

    tr_loader  = DataLoader(AlarmDataset(X_f_tr, y_f_tr),
                            batch_size=best_params.get('batch_size', 32),
                            shuffle=True)
    val_loader = DataLoader(AlarmDataset(X_f_val, y_f_val),
                            batch_size=best_params.get('batch_size', 32))

    model = make_model(best_params, INPUT_SIZE)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=best_params.get('lr', 1e-3))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5)
    criterion = FocalLoss(
        gamma=2.0, pos_weight=best_params.get('pos_weight', POS_WEIGHT_DEFAULT))

    best_val_prauc = 0.0
    patience_ctr   = 0
    best_state     = None
    history        = {'train_loss': [], 'val_loss': [], 'val_prauc': []}

    for epoch in range(EPOCHS):
        tr_loss = train_one_epoch(model, tr_loader, optimizer, criterion)
        val_loss, val_probs, val_labels = evaluate(model, val_loader)
        val_score = pr_auc(val_probs, val_labels)
        scheduler.step(val_score)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(val_loss)
        history['val_prauc'].append(val_score)

        if val_score > best_val_prauc:
            best_val_prauc = val_score
            best_state = {k: v.cpu().clone()
                         for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1

        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1:3d} | '
                  f'Train Loss: {tr_loss:.4f} | '
                  f'Val Loss: {val_loss:.4f} | '
                  f'Val PR-AUC: {val_score:.4f}')

        if patience_ctr >= PATIENCE:
            print(f'  Early stop at epoch {epoch + 1}')
            break

    # Reload best weights and find best threshold on this val fold
    model.load_state_dict(best_state)
    _, val_probs_best, val_labels_best = evaluate(model, val_loader)
    thr = best_threshold_for_precision(val_probs_best, val_labels_best, min_recall=0.6)

    fold_results.append({
        'fold'        : fold_idx + 1,
        'val_pr_auc'  : best_val_prauc,
        'threshold'   : thr,
        'val_probs'   : val_probs_best,
        'val_labels'  : val_labels_best,
    })
    fold_histories.append(history)
    best_models.append(best_state)

    print(f'  Best Val PR-AUC: {best_val_prauc:.4f} | Threshold: {thr:.2f}')

print('\n── 10-Fold CV Complete ──────────────────────────────────')
mean_prauc = np.mean([r['val_pr_auc'] for r in fold_results])
std_prauc  = np.std([r['val_pr_auc']  for r in fold_results])
print(f'Mean Val PR-AUC: {mean_prauc:.4f} ± {std_prauc:.4f}')


── Fold 1/10 ──────────────────────


NameError: name 'best_params' is not defined

## 10. Loss Curves — All Folds

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 8))
axes = axes.flatten()

for i, (hist, res) in enumerate(zip(fold_histories, fold_results)):
    ax = axes[i]
    epochs_ran = range(1, len(hist['train_loss']) + 1)
    ax.plot(epochs_ran, hist['train_loss'], label='Train Loss', color='steelblue')
    ax.plot(epochs_ran, hist['val_loss'],   label='Val Loss',   color='tomato')
    ax2 = ax.twinx()
    ax2.plot(epochs_ran, hist['val_prauc'], label='Val PR-AUC',
             color='forestgreen', linestyle='--', alpha=0.8)
    ax2.set_ylabel('PR-AUC', color='forestgreen', fontsize=8)
    ax2.tick_params(axis='y', labelcolor='forestgreen')
    ax.set_title(f'Fold {res["fold"]}  (PR-AUC={res["val_pr_auc"]:.3f})', fontsize=9)
    ax.set_xlabel('Epoch', fontsize=8)
    ax.set_ylabel('Loss', fontsize=8)
    ax.legend(fontsize=7, loc='upper right')

plt.suptitle('GRU Training — Loss & Val PR-AUC per Fold', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('fold_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fold_loss_curves.png')

## 11. Per-Fold Metrics Summary

In [ ]:
summary_rows = []
for res in fold_results:
    thr   = res['threshold']
    preds = (res['val_probs'] >= thr).astype(int)
    labels = res['val_labels'].astype(int)
    report = classification_report(labels, preds, output_dict=True, zero_division=0)
    pos = report.get('1', report.get('1.0', {}))
    summary_rows.append({
        'Fold'      : res['fold'],
        'Threshold' : round(thr, 2),
        'PR-AUC'    : round(res['val_pr_auc'], 4),
        'Precision' : round(pos.get('precision', 0), 4),
        'Recall'    : round(pos.get('recall', 0), 4),
        'F1'        : round(pos.get('f1-score', 0), 4),
        'Accuracy'  : round(report['accuracy'], 4),
    })

summary_df = pd.DataFrame(summary_rows)
mean_row   = summary_df.drop(columns='Fold').mean().round(4).to_dict()
mean_row['Fold'] = 'MEAN'
std_row    = summary_df.drop(columns='Fold').std().round(4).to_dict()
std_row['Fold']  = 'STD'
summary_df = pd.concat([summary_df,
                         pd.DataFrame([mean_row, std_row])],
                        ignore_index=True)
print(summary_df.to_string(index=False))

## 12. Per-Fold Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 8))
axes = axes.flatten()

for i, res in enumerate(fold_results):
    thr   = res['threshold']
    preds = (res['val_probs'] >= thr).astype(int)
    cm    = confusion_matrix(res['val_labels'].astype(int), preds)
    disp  = ConfusionMatrixDisplay(cm, display_labels=['No Alarm', 'Alarm'])
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(f'Fold {res["fold"]}  (thr={thr:.2f})', fontsize=9)
    axes[i].set_xlabel('Predicted', fontsize=8)
    axes[i].set_ylabel('Actual', fontsize=8)

plt.suptitle('Confusion Matrices — All 10 Folds (Val Set)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('fold_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fold_confusion_matrices.png')

## 13. Precision-Recall Curves — All Folds

In [ ]:
plt.figure(figsize=(9, 7))
colors = plt.cm.tab10(np.linspace(0, 1, N_FOLDS))

for res, color in zip(fold_results, colors):
    prec, rec, _ = precision_recall_curve(res['val_labels'], res['val_probs'])
    plt.plot(rec, prec, color=color, alpha=0.7,
             label=f'Fold {res["fold"]} (AUC={res["val_pr_auc"]:.3f})')

plt.axhline(y=0.6, color='grey', linestyle=':', alpha=0.6, label='Recall=0.6 threshold')
plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curves — 10-Fold CV', fontsize=14)
plt.legend(fontsize=8, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: pr_curves.png')

## 14. Final Model — Train on Full Train Set, Evaluate on Validation & Test

In [ ]:
# Use median threshold from CV folds
final_threshold = float(np.median([r['threshold'] for r in fold_results]))
print(f'Final threshold (median of CV folds): {final_threshold:.2f}')

# Train final model
full_tr_loader  = DataLoader(AlarmDataset(X_train_sc, y_train),
                              batch_size=best_params.get('batch_size', 32),
                              shuffle=True)
val_loader_full = DataLoader(AlarmDataset(X_val_sc, y_val),
                              batch_size=best_params.get('batch_size', 32))
test_loader     = DataLoader(AlarmDataset(X_test_sc, y_test),
                              batch_size=best_params.get('batch_size', 32))

final_model = make_model(best_params, INPUT_SIZE)
optimizer   = torch.optim.Adam(
    final_model.parameters(), lr=best_params.get('lr', 1e-3))
scheduler   = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5)
criterion   = FocalLoss(
    gamma=2.0, pos_weight=best_params.get('pos_weight', POS_WEIGHT_DEFAULT))

best_val_prauc  = 0.0
patience_ctr    = 0
final_best_state = None
final_history   = {'train_loss': [], 'val_loss': [], 'val_prauc': []}

print('Training final model...')
for epoch in range(EPOCHS):
    tr_loss = train_one_epoch(final_model, full_tr_loader, optimizer, criterion)
    val_loss, val_probs, val_labels = evaluate(final_model, val_loader_full)
    score = pr_auc(val_probs, val_labels)
    scheduler.step(score)

    final_history['train_loss'].append(tr_loss)
    final_history['val_loss'].append(val_loss)
    final_history['val_prauc'].append(score)

    if score > best_val_prauc:
        best_val_prauc = score
        final_best_state = {k: v.cpu().clone()
                           for k, v in final_model.state_dict().items()}
        patience_ctr = 0
    else:
        patience_ctr += 1

    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1:3d} | '
              f'Train Loss: {tr_loss:.4f} | '
              f'Val Loss: {val_loss:.4f} | '
              f'Val PR-AUC: {score:.4f}')

    if patience_ctr >= PATIENCE:
        print(f'  Early stop at epoch {epoch + 1}')
        break

final_model.load_state_dict(final_best_state)
print(f'\nBest Val PR-AUC (final model): {best_val_prauc:.4f}')

## 15. Final Model — Loss Curve

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))
ep = range(1, len(final_history['train_loss']) + 1)
ax1.plot(ep, final_history['train_loss'], label='Train Loss', color='steelblue')
ax1.plot(ep, final_history['val_loss'],   label='Val Loss',   color='tomato')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend(loc='upper left')

ax2 = ax1.twinx()
ax2.plot(ep, final_history['val_prauc'], label='Val PR-AUC',
         color='forestgreen', linestyle='--')
ax2.set_ylabel('PR-AUC', color='forestgreen')
ax2.tick_params(axis='y', labelcolor='forestgreen')
ax2.legend(loc='upper right')

plt.title('Final Model — Training Curve')
plt.tight_layout()
plt.savefig('final_model_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: final_model_loss_curve.png')

## 16. Validation Set — Full Evaluation

In [ ]:
_, val_probs_final, val_labels_final = evaluate(final_model, val_loader_full)

# Fine-tune threshold on actual validation set
val_threshold = best_threshold_for_precision(
    val_probs_final, val_labels_final, min_recall=0.6)
print(f'Validation-tuned threshold: {val_threshold:.2f}')

val_preds = (val_probs_final >= val_threshold).astype(int)

print('\n── Validation Classification Report ─────────────────────')
print(classification_report(val_labels_final.astype(int), val_preds,
                             target_names=['No Alarm', 'Alarm']))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
cm_val = confusion_matrix(val_labels_final.astype(int), val_preds)
ConfusionMatrixDisplay(cm_val, display_labels=['No Alarm', 'Alarm']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Validation Confusion Matrix (thr={val_threshold:.2f})')

# PR Curve
prec_v, rec_v, _ = precision_recall_curve(val_labels_final, val_probs_final)
prauc_v = auc(rec_v, prec_v)
axes[1].plot(rec_v, prec_v, color='steelblue', lw=2)
axes[1].fill_between(rec_v, prec_v, alpha=0.15, color='steelblue')
axes[1].set_title(f'Validation PR Curve (AUC={prauc_v:.4f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('validation_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: validation_evaluation.png')

## 17. Test Set — Final Evaluation

In [ ]:
_, test_probs, test_labels = evaluate(final_model, test_loader)
test_preds = (test_probs >= val_threshold).astype(int)

print('\n── TEST SET Classification Report ───────────────────────')
print(classification_report(test_labels.astype(int), test_preds,
                             target_names=['No Alarm', 'Alarm']))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
cm_test = confusion_matrix(test_labels.astype(int), test_preds)
ConfusionMatrixDisplay(cm_test, display_labels=['No Alarm', 'Alarm']).plot(
    ax=axes[0], colorbar=False, cmap='Oranges')
axes[0].set_title(f'Test Confusion Matrix (thr={val_threshold:.2f})')

# PR Curve
prec_t, rec_t, _ = precision_recall_curve(test_labels, test_probs)
prauc_t = auc(rec_t, prec_t)
axes[1].plot(rec_t, prec_t, color='darkorange', lw=2)
axes[1].fill_between(rec_t, prec_t, alpha=0.15, color='darkorange')
axes[1].set_title(f'Test PR Curve (AUC={prauc_t:.4f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('test_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: test_evaluation.png')

## 18. Save Model & Scaler

In [ ]:
import joblib

torch.save({
    'model_state_dict': final_best_state,
    'best_params'     : best_params,
    'threshold'       : val_threshold,
    'input_size'      : INPUT_SIZE,
    'lookback'        : LOOKBACK,
    'feature_cols'    : FEATURE_COLS,
}, 'gru_alarm_model.pt')

joblib.dump(scaler_global, 'scaler.pkl')

print('Saved: gru_alarm_model.pt')
print('Saved: scaler.pkl')
print(f'\nFinal threshold applied at inference: {val_threshold:.2f}')
print(f'Best hyperparameters: {best_params}')

In [ ]:
# ── 19. Inference Pipeline (Testing on 13 Columns) ──────────────────────────
# Requirement: "testing should be on only 13 columns"
# This function demonstrates how to take a raw 13-column DataFrame, compute the
# required 93 features dynamically, scale them, and produce predictions.

def run_inference_pipeline(raw_data_13_cols, selected_features, scaler, model, lookback, device):
    """
    Args:
        raw_data_13_cols: DataFrame containing the 13 base sensor columns + timestamp
        selected_features: List of 93 feature names the model was trained on
        scaler: The fitted StandardScaler from training
        model: The trained GRU model
        lookback: The sequence lookback length (e.g., 20)
    Returns:
        DataFrame with predictions
    """
    # 1. Feature Engineering (Dynamically compute only what's needed)
    base_cols = [c for c in raw_data_13_cols.columns if c != 'timestamp']
    df_eng = create_engineered_features(raw_data_13_cols, base_cols, selected_features)
    
    # Ensure all selected features exist (fill missing with 0 or mean if any edge cases)
    for f in selected_features:
        if f not in df_eng.columns:
            df_eng[f] = 0.0
            
    # Extract feature matrix
    X_features = df_eng[selected_features].values.astype(np.float32)
    
    # Replace infs and nans (similar to clean_sequences)
    X_features[~np.isfinite(X_features)] = np.nan
    col_medians = np.nanmedian(X_features, axis=0)
    for i in range(X_features.shape[1]):
        mask = ~np.isfinite(X_features[:, i])
        if mask.any():
            X_features[mask, i] = col_medians[i] if not np.isnan(col_medians[i]) else 0.0
            
    # Scale features
    X_scaled = scaler.transform(X_features)
    
    # Build sequences (assuming stride=1 for testing)
    sequences = []
    timestamps = []
    
    for i in range(len(X_scaled) - lookback + 1):
        sequences.append(X_scaled[i:i + lookback])
        if 'timestamp' in raw_data_13_cols.columns:
            timestamps.append(raw_data_13_cols['timestamp'].iloc[i + lookback - 1])
        
    if not sequences:
        print("Not enough data to form a single sequence.")
        return pd.DataFrame()
        
    X_tensor = torch.tensor(np.array(sequences), dtype=torch.float32).to(device)
    
    # Predict
    model.eval()
    with torch.no_grad():
        logits = model(X_tensor)
        probs = torch.sigmoid(logits).cpu().numpy()
        
    # Format output
    out_df = pd.DataFrame({'alarm_probability': probs})
    if timestamps:
        out_df['timestamp'] = timestamps
        out_df = out_df[['timestamp', 'alarm_probability']]
        
    # Apply final threshold (global variable from notebook)
    try:
        out_df['alarm_prediction'] = (probs >= val_threshold).astype(int)
    except NameError:
        out_df['alarm_prediction'] = (probs >= 0.6).astype(int) # fallback
        
    return out_df

# Example Usage (Uncomment to test):
# raw_test_data = df[BASE_COLS + ['timestamp']].tail(100).copy() # Simulating a 13-col raw input
# predictions = run_inference_pipeline(raw_test_data, SELECTED_FEATURES, scaler_global, final_model, LOOKBACK, DEVICE)
# print(predictions.head())


In [ ]:
# ── 19. Inference Pipeline (Testing on 13 Columns) ──────────────────────────
# Requirement: "testing should be on only 13 columns"
# This function demonstrates how to take a raw 13-column DataFrame, compute the
# required 93 features dynamically, scale them, and produce predictions.

def run_inference_pipeline(raw_data_13_cols, selected_features, scaler, model, lookback, device):
    """
    Args:
        raw_data_13_cols: DataFrame containing the 13 base sensor columns + timestamp
        selected_features: List of 93 feature names the model was trained on
        scaler: The fitted StandardScaler from training
        model: The trained GRU model
        lookback: The sequence lookback length (e.g., 20)
    Returns:
        DataFrame with predictions
    """
    # 1. Feature Engineering (Dynamically compute only what's needed)
    base_cols = [c for c in raw_data_13_cols.columns if c != 'timestamp']
    df_eng = create_engineered_features(raw_data_13_cols, base_cols, selected_features)
    
    # Ensure all selected features exist (fill missing with 0 or mean if any edge cases)
    for f in selected_features:
        if f not in df_eng.columns:
            df_eng[f] = 0.0
            
    # Extract feature matrix
    X_features = df_eng[selected_features].values.astype(np.float32)
    
    # Replace infs and nans (similar to clean_sequences)
    X_features[~np.isfinite(X_features)] = np.nan
    col_medians = np.nanmedian(X_features, axis=0)
    for i in range(X_features.shape[1]):
        mask = ~np.isfinite(X_features[:, i])
        if mask.any():
            X_features[mask, i] = col_medians[i] if not np.isnan(col_medians[i]) else 0.0
            
    # Scale features
    X_scaled = scaler.transform(X_features)
    
    # Build sequences (assuming stride=1 for testing)
    sequences = []
    timestamps = []
    
    for i in range(len(X_scaled) - lookback + 1):
        sequences.append(X_scaled[i:i + lookback])
        if 'timestamp' in raw_data_13_cols.columns:
            timestamps.append(raw_data_13_cols['timestamp'].iloc[i + lookback - 1])
        
    if not sequences:
        print("Not enough data to form a single sequence.")
        return pd.DataFrame()
        
    X_tensor = torch.tensor(np.array(sequences), dtype=torch.float32).to(device)
    
    # Predict
    model.eval()
    with torch.no_grad():
        logits = model(X_tensor)
        probs = torch.sigmoid(logits).cpu().numpy()
        
    # Format output
    out_df = pd.DataFrame({'alarm_probability': probs})
    if timestamps:
        out_df['timestamp'] = timestamps
        out_df = out_df[['timestamp', 'alarm_probability']]
        
    # Apply final threshold (global variable from notebook)
    try:
        out_df['alarm_prediction'] = (probs >= val_threshold).astype(int)
    except NameError:
        out_df['alarm_prediction'] = (probs >= 0.6).astype(int) # fallback
        
    return out_df

# Example Usage (Uncomment to test):
# raw_test_data = df[BASE_COLS + ['timestamp']].tail(100).copy() # Simulating a 13-col raw input
# predictions = run_inference_pipeline(raw_test_data, SELECTED_FEATURES, scaler_global, final_model, LOOKBACK, DEVICE)
# print(predictions.head())
